In [32]:
from master_table import load_table
master = load_table()

In [ ]:
#@title Ecos
import ecos

In [29]:
import requests
import pandas as pd
import xml.etree.ElementTree as ET

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7",
}

url ='https://ecos.bok.or.kr/api/StatisticTableList/K6KQJ2MTT16VOJRPLIMJ/json/kr/1/1000/'
response = requests.get(url, headers=HEADERS, timeout=60)
StatisticTable = pd.DataFrame(response.json()['StatisticTableList']['row'])
StatisticTable = StatisticTable[StatisticTable['CYCLE'].notna()]
StatisticTable

,P_STAT_CODE,STAT_CODE,STAT_NAME,CYCLE,SRCH_YN,ORG_NAME
4,0000000622,102Y004,"1.1.1.1.1. 본원통화 구성내역(평잔, 계절조정계열)",M,Y,NaN
5,0000000622,102Y002,"1.1.1.1.2. 본원통화 구성내역(평잔, 원계열)",M,Y,NaN
6,0000000622,102Y003,"1.1.1.1.3. 본원통화 구성내역(말잔, 계절조정계열)",M,Y,NaN
7,0000000622,102Y001,"1.1.1.1.4. 본원통화 구성내역(말잔, 원계열)",M,Y,한국은행
10,0000000624,161Y001,"1.1.2.1.1. M1 상품별 구성내역(평잔, 계절조정계열)",M,Y,NaN
...,...,...,...,...,...,...
828,0000000447,251Y003,9.2.1.1. 총량,A,Y,NaN
829,0000000447,251Y002,9.2.1.2. 한국/북한 배율,A,Y,NaN
830,0000000396,251Y001,9.2.2. 북한의 경제활동별 국내총생산,A,Y,한국은행
832,0000000704,252Y001,9.2.3.1. 시장물가지수,Q,Y,한국은행


In [98]:
import importlib
import master_table
importlib.reload(master_table)
fetch_one(master, dataset_id="102Y001")

,STAT_CODE,STAT_NAME,ITEM_CODE1,ITEM_NAME1,ITEM_CODE2,ITEM_NAME2,ITEM_CODE3,ITEM_NAME3,ITEM_CODE4,ITEM_NAME4,UNIT_NAME,WGT,TIME,DATA_VALUE
0,102Y001,"1.1.1.1.4. 본원통화 구성내역(말잔, 원계열)",ABA2,"본원통화(말잔, 원계열)",None,None,None,None,None,None,십억원,None,200310,37234.4
1,102Y001,"1.1.1.1.4. 본원통화 구성내역(말잔, 원계열)",ABA203,현금통화,None,None,None,None,None,None,십억원,None,200310,17666.5
2,102Y001,"1.1.1.1.4. 본원통화 구성내역(말잔, 원계열)",ABA204,중앙은행 대 예금취급기관부채,None,None,None,None,None,None,십억원,None,200310,19568
3,102Y001,"1.1.1.1.4. 본원통화 구성내역(말잔, 원계열)",ABA2,"본원통화(말잔, 원계열)",None,None,None,None,None,None,십억원,None,200311,36423.1
4,102Y001,"1.1.1.1.4. 본원통화 구성내역(말잔, 원계열)",ABA203,현금통화,None,None,None,None,None,None,십억원,None,200311,17946.8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
760,102Y001,"1.1.1.1.4. 본원통화 구성내역(말잔, 원계열)",ABA203,현금통화,None,None,None,None,None,None,십억원,None,202411,179443.4
761,102Y001,"1.1.1.1.4. 본원통화 구성내역(말잔, 원계열)",ABA204,중앙은행 대 예금취급기관부채,None,None,None,None,None,None,십억원,None,202411,98815.3
762,102Y001,"1.1.1.1.4. 본원통화 구성내역(말잔, 원계열)",ABA2,"본원통화(말잔, 원계열)",None,None,None,None,None,None,십억원,None,202412,283486.9
763,102Y001,"1.1.1.1.4. 본원통화 구성내역(말잔, 원계열)",ABA203,현금통화,None,None,None,None,None,None,십억원,None,202412,182070.2


In [132]:
#@title Enara
import requests
import pandas as pd
import xml.etree.ElementTree as ET

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept": "text/xml, application/xml, */*",
    "Accept-Language": "ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7",
}

url = 'https://www.index.go.kr/unity/openApi/xml_idx.do?userId=apjh2529&idntfcId=229151DF51U114T0'
response = requests.get(url, headers=HEADERS, timeout=60)


# XML 파싱
root = ET.fromstring(response.text)

rows = []
for indicator in root.findall("지표"):
    base = {
        "지표체계": indicator.findtext("지표체계"),
        "지표코드": indicator.findtext("지표코드"),
        "지표명": indicator.findtext("지표명"),
        "수치수정일": indicator.findtext("수치수정일"),
        "의미분석수정일": indicator.findtext("의미분석수정일"),
        "지표조회수": indicator.findtext("지표조회수"),
    }

    # 통계표 여러 개면 각각 row로 확장
    for table in indicator.findall("통계표"):
        row = base.copy()
        row["통계표코드"] = table.findtext("통계표코드")
        row["통계표명"] = table.findtext("통계표명")
        row["통계표구분"] = table.findtext("통계표구분")
        rows.append(row)

# DataFrame 생성
df_enara_table = pd.DataFrame(rows)

In [133]:
df_enara_table

,지표체계,지표코드,지표명,수치수정일,의미분석수정일,지표조회수,통계표코드,통계표명,통계표구분
0,e-나라지표,2455,10대 수출입 품목,20251031,20200626,573,245501,10대 수출품목,기본통계표
1,e-나라지표,2455,10대 수출입 품목,20251031,20200626,573,245502,10대 수입품목,기본통계표
2,e-나라지표,1609,112신고접수 현황,20260401,20200828,171,160901,112접수건수 및 현장평균도착시간,기본통계표
3,e-나라지표,1601,1388 청소년 전화 접수현황,20260203,20200901,216,160101,1388 청소년전화 접수현황,기본통계표
4,e-나라지표,1728,"1심, 2심 무죄 현황",20260129,20200824,260,172801,"1심, 2심 무죄현황",기본통계표
...,...,...,...,...,...,...,...,...,...
1069,국민삶의질지표,8050,안전에 대한 전반적 인식,20250103,20201123,69,805002,안전에 대한 전반적 인식,기본통계표
1070,국민삶의질지표,8051,야간보행안전도,20221201,20201130,11,805101,야간보행 안전도,기본통계표
1071,국민삶의질지표,8098,토양환경 만족도,20250103,20191106,29,809802,토양환경 만족도,기본통계표
1072,국민삶의질지표,8030,학교교육 효과,20250103,20191106,54,803002,학교교육 효과,기본통계표


In [61]:
import importlib
import master_table
importlib.reload(master_table)
fetch_one(master, dataset_id="172801")

ValueError: No row found for table_id/index=172801

In [6]:
#@title imf
import requests
from imf import imf_indicators_to_dataframe
r = requests.get("https://www.imf.org/external/datamapper/api/v1/indicators", timeout=60)
r.raise_for_status()
df = imf_indicators_to_dataframe(r.json())

In [46]:
#@title kosis
import pandas as pd

file_path = "주제별통계.xls"  # 필요시 xlsx

# 1) 실제 시트명 추출
xls = pd.ExcelFile(file_path, engine="xlrd")
sheet_names = xls.sheet_names
print(sheet_names)

# 2) 첫 번째 시트를 컬럼 기준으로 사용
base_sheet = sheet_names[0]
df_base = pd.read_excel(file_path, sheet_name=base_sheet, engine="xlrd", header=3)
base_cols = df_base.columns

# 3) 모든 시트 순회하며 sheet1(=첫 시트) 컬럼에 맞춰 concat
dfs = [df_base]
for s in sheet_names[1:]:
    d = pd.read_excel(file_path, sheet_name=s, engine="xlrd")
    d = d.reindex(columns=base_cols)
    dfs.append(d)

data_kosis = pd.concat(dfs, ignore_index=True)
data_kosis


['Sheet1', 'Sheet2', 'Sheet3', 'Sheet4', 'Sheet5']


,Level,통계명,통계표조회,출처,수록기간,경로,경로(LIST_ID),통계표 아이디(TBL_ID)
0,1.0,인구,NaN,NaN,NaN,주제별통계 > 인구,> A,NaN
1,2.0,인구총조사,NaN,NaN,NaN,주제별통계 > 인구 > 인구총조사,> A > A_4,NaN
2,3.0,인구부문,NaN,NaN,NaN,주제별통계 > 인구 > 인구총조사 > 인구부문,> A > A_4 > A11,NaN
3,4.0,총조사인구(2015년 이후),NaN,NaN,NaN,주제별통계 > 인구 > 인구총조사 > 인구부문 > 총조사인구(2015년 이후),> A > A_4 > A11 > A11_2015_1,NaN
4,5.0,"전수부문 (등록센서스, 2015년 이후)",NaN,NaN,NaN,주제별통계 > 인구 > 인구총조사 > 인구부문 > 총조사인구(2015년 이후) > ...,> A > A_4 > A11 > A11_2015_1 > A11_2015_1_10,NaN
...,...,...,...,...,...,...,...,...
290809,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
290810,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
290811,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
290812,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [53]:
data_kosis_filtered

,통계명,출처,수록기간,통계표 아이디(TBL_ID),start_date,end_date,cycle
6,"인구, 가구 및 주택 – 읍면...","국가데이터처,「인구총조사」",년 (2015 ~ 2024),DT_1IN1502,2015,2024,Y
7,연령 및 성별 인구 – 읍면동,"국가데이터처,「인구총조사」",년 (2015 ~ 2024),DT_1IN1503,2015,2024,Y
8,"주요 인구지표(부양비, 노령화...","국가데이터처,「인구총조사」",년 (2015 ~ 2024),DT_1IN2030,2015,2024,Y
9,"성, 연령 및 가구주와의 관계...","국가데이터처,「인구총조사」",년 (2015 ~ 2024),DT_1IN1507,2015,2024,Y
10,"성, 연령 및 세대구성별 인구...","국가데이터처,「인구총조사」",년 (2015 ~ 2024),DT_1IN1509,2015,2024,Y
...,...,...,...,...,...,...,...
64991,방조제축조사업,"한국농어촌공사,「농업생산기반정비통계조사」",년 (1987 ~ 1992),TX_31101_A129,1987,1992,Y
64992,배수개선,"한국농어촌공사,「농업생산기반정비통계조사」",년 (1975 ~ 1992),TX_31101_A118,1975,1992,Y
64993,생산기반확충사업,"한국농어촌공사,「농업생산기반정비통계조사」",년 (1945 ~ 1992),TX_31101_A116,1945,1992,Y
64994,소규모(시도),"한국농어촌공사,「농업생산기반정비통계조사」",년 (1965 ~ 1992),TX_31101_A125,1965,1992,Y


In [51]:
import re
import pandas as pd
import numpy as np

data_kosis_filtered = data_kosis.drop_duplicates(subset=['통계표 아이디(TBL_ID)']).dropna(subset=['통계표 아이디(TBL_ID)'])[['통계명','출처','수록기간','통계표 아이디(TBL_ID)']]

_CYCLE_PREFIXES = [
    ("격년", "F"),
    ("격별", "F"),
    ("분기", "Q"),
    ("반기", "H"),
    ("년", "Y"),
    ("월", "M"),
    ("격", "F"),
]
def parse_surok_gigan_raw(val):
    """괄호 안 구간만 분해. start/end는 문자열 그대로."""
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return None, None, None
    s = str(val).strip()
    if not s:
        return None, None, None
    cycle = None
    for prefix, code in _CYCLE_PREFIXES:
        if s.startswith(prefix):
            cycle = code
            s = s[len(prefix) :].lstrip()
            break
    m = re.search(r"[\(（]\s*([^)）]+)\s*[\)）]", s)
    if not m:
        return None, None, cycle
    parts = [p.strip() for p in re.split(r"\s*[~～－]\s*", m.group(1)) if p.strip()]
    if not parts:
        return None, None, cycle
    return parts[0], parts[-1], cycle
def add_surok_columns(df, col="수록기간"):
    out = df.copy()
    parsed = out[col].map(parse_surok_gigan_raw)
    out["start_date"] = [a for a, b, c in parsed]
    out["end_date"] = [b for a, b, c in parsed]
    out["cycle"] = [c for a, b, c in parsed]
    return out


# 사용
data_kosis_filtered = add_surok_columns(data_kosis_filtered, "수록기간")

In [35]:
#@title 종합
import requests
import pandas as pd
from bs4 import BeautifulSoup
from tqdm import tqdm
import xml.etree.ElementTree as ET

# HTML 데이터 파싱
html = """
<ul class="listCate"><li data-toggle="tab" class="litest" id="0120000"><a href="#" name="0120000"><i class="ico-category category-01"></i><span>국토/도시</span><em>(97)</em></a></li><li data-toggle="tab" class="litest active" id="0150000"><a href="#" name="0150000"><i class="ico-category category-02"></i><span>주택</span><em>(368)</em></a></li><li data-toggle="tab" class="litest" id="0590000"><a href="#" name="0590000"><i class="ico-category category-03"></i><span>토지</span><em>(201)</em></a></li><li data-toggle="tab" class="litest" id="0130000"><a href="#" name="0130000"><i class="ico-category category-04"></i><span>건설</span><em>(178)</em></a></li><li data-toggle="tab" class="litest" id="0160000"><a href="#" name="0160000"><i class="ico-category category-05"></i><span>교통/물류</span><em>(143)</em></a></li><li data-toggle="tab" class="litest" id="0170000"><a href="#" name="0170000"><i class="ico-category category-06"></i><span>항공</span><em>(17)</em></a></li><li data-toggle="tab" class="litest" id="0180000"><a href="#" name="0180000"><i class="ico-category category-07"></i><span>도로/철도</span><em>(79)</em></a></li></ul>
"""
soup = BeautifulSoup(html, 'html.parser')

# 모든 'codeCate' ID 추출
codeCate_list = [li.attrs['id'] for li in soup.find_all('li')]

# 1. 전체 통계 리스트 수집
stat_url = "https://stat.molit.go.kr/portal/cate/partSttsAjx.do"
df_list = []

for codeCate in tqdm(codeCate_list, desc="수집 중: 통계 리스트"):
    params = {
        'actionFlag': 'mid',
        'statGb': '0310001',
        'codeCate': codeCate,
        'iconFlag': 'false'
    }

    response = requests.post(stat_url, params=params)
    lxml = BeautifulSoup(response.content, 'lxml')

    # statlist 내의 모든 <a> 태그 찾기
    stat_entries = lxml.find("statlist").find_all("a") if lxml.find("statlist") else []

    # 데이터 저장
    for entry in stat_entries:
        rsid = entry.get("rsid")
        stat_gb = entry.get("stat-gb")
        name = entry.text.strip()

        df_list.append({"codeCate": codeCate, "rsid": rsid, "stat_gb": stat_gb, "name": name})

# DataFrame 생성
df = pd.DataFrame(df_list)

# 2. 세부 데이터 (data_detail) 수집
data_detail_list = []
base_url = "https://stat.molit.go.kr"

for index, row in tqdm(df.iterrows(), total=len(df), desc="수집 중: 세부 데이터"):
    params = {
        'actionFlag': 'statform',
        'statGb': '0310001',
        'codeCate': row['codeCate'],
        'codeDetail': row['rsid'],
        'iconFlag': 'true'
    }

    response = requests.post(stat_url, params=params)
    lxml = BeautifulSoup(response.content, 'lxml')

    # statlist 내의 모든 <a> 태그 찾기
    stat_entries = lxml.find("statlist").find_all("a") if lxml.find("statlist") else []

    # 데이터 저장
    for entry in stat_entries:
        href = entry.get("href")
        name = entry.text.strip()

        data_detail_list.append({
            "codeCate": row['codeCate'],
            "rsid": row['rsid'],
            "name": name,
            "url": base_url + href
        })

# DataFrame 생성
df_detail = pd.DataFrame(data_detail_list)

수집 중: 통계 리스트:   0%|          | 0/7 [00:00<?, ?it/s]C:\Users\apjh2\AppData\Local\Temp\ipykernel_24304\796857088.py:30: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  lxml = BeautifulSoup(response.content, 'lxml')
수집 중: 세부 데이터:   0%|          | 0/67 [00:00<?, ?it/s]C:\Users\apjh2\AppData\Local\Temp\ipykernel_24304\79685708

In [36]:
df_detail

,codeCate,rsid,name,url
0,0120000,19,면적별 건축물 현황태극마크,https://stat.molit.go.kr/portal/cate/viewChk.d...
1,0120000,19,면적별 건축물 현황(50만_이상_도시)태극마크,https://stat.molit.go.kr/portal/cate/viewChk.d...
2,0120000,19,소유구분별 건축물 현황태극마크,https://stat.molit.go.kr/portal/cate/viewChk.d...
3,0120000,19,소유구분별 건축물 현황(50만_이상_도시)태극마크,https://stat.molit.go.kr/portal/cate/viewChk.d...
4,0120000,19,용도별 건축물 현황태극마크,https://stat.molit.go.kr/portal/cate/viewChk.d...
...,...,...,...,...
1185,0180000,542,철도 차종별 보유현황태극마크,https://stat.molit.go.krhttps://kosis.kr/statH...
1186,0180000,542,철도관련 통신 시설물 현황태극마크,https://stat.molit.go.krhttps://kosis.kr/statH...
1187,0180000,542,철도총괄지표(지역간철도)태극마크,https://stat.molit.go.krhttps://kosis.kr/statH...
1188,0180000,542,철도화물 수송실적(톤수)태극마크,https://stat.molit.go.krhttps://kosis.kr/statH...


In [39]:
import re

# hFormId 값을 저장할 리스트 생성
hFormId_list = []

# URL에서 hFormId 값을 추출하여 리스트에 추가
for url in df_detail['url']:
    match = re.search(r"hFormId=(\d+)", url)
    if match:
        hFormId_list.append(match.group(1))  # 숫자 값만 저장
    else:
        hFormId_list.append(None)  # 값이 없는 경우 None 저장

# DataFrame에 새로운 컬럼 추가
df_detail['hFormId'] = hFormId_list
df_detail_filtered = df_detail.dropna()
df_detail_filtered

,codeCate,rsid,name,url,hFormId
0,0120000,19,면적별 건축물 현황태극마크,https://stat.molit.go.kr/portal/cate/viewChk.d...,540
1,0120000,19,면적별 건축물 현황(50만_이상_도시)태극마크,https://stat.molit.go.kr/portal/cate/viewChk.d...,5391
2,0120000,19,소유구분별 건축물 현황태극마크,https://stat.molit.go.kr/portal/cate/viewChk.d...,560
3,0120000,19,소유구분별 건축물 현황(50만_이상_도시)태극마크,https://stat.molit.go.kr/portal/cate/viewChk.d...,5392
4,0120000,19,용도별 건축물 현황태극마크,https://stat.molit.go.kr/portal/cate/viewChk.d...,522
...,...,...,...,...,...
1159,0180000,68,7-3.가출현황_가출인발견(종료)태극마크,https://stat.molit.go.kr/portal/cate/viewChk.d...,894
1160,0180000,67,건널목사고 상세현황태극마크,https://stat.molit.go.kr/portal/cate/viewChk.d...,876
1161,0180000,67,사상사고 상세현황태극마크,https://stat.molit.go.kr/portal/cate/viewChk.d...,878
1162,0180000,67,열차사고 상세현황태극마크,https://stat.molit.go.kr/portal/cate/viewChk.d...,874


In [40]:
#@title StyleNum, Dt_list 가져오기
from tqdm import tqdm
import requests
from bs4 import BeautifulSoup

style_list = []
final_dt_list = []

for index, row in tqdm(df_detail_filtered.iterrows(), total=df_detail_filtered.shape[0]):
    rsid = row['rsid']
    hFormId = row['hFormId']
    url = f'https://stat.molit.go.kr/portal/cate/statView.do?hRsId={rsid}&hFormId={hFormId}&hDivEng=&month_yn='

    try:
        response = requests.get(url)
        response.raise_for_status()  # HTTP 오류가 발생하면 예외 발생
        soup = BeautifulSoup(response.content, 'html.parser')

        # 스타일 번호 가져오기
        try:
            style = soup.find('select', {'id': 'sStyleNum'})
            selected_option = style.find("option", {"selected": True}) if style else None
            style_num = selected_option["value"] if selected_option else None
        except AttributeError:
            style_num = None
            print(f"[Warning] 'sStyleNum' 관련 AttributeError 발생 (index: {index})")

        # 날짜 목록 가져오기
        try:
            dt = soup.find('select', {'id': 'sStart'})
            dt_list = [i.text for i in dt.find_all('option')] if dt else []
        except AttributeError:
            dt_list = []
            print(f"[Warning] 'sStart' 관련 AttributeError 발생 (index: {index})")

        style_list.append(style_num)
        final_dt_list.append(dt_list)

    except requests.RequestException as e:
        print(f"[Error] 요청 실패 (index: {index}, URL: {url}): {e}")
        style_list.append(None)
        final_dt_list.append([])

print("데이터 수집 완료.")

 81%|████████  | 670/832 [12:26<02:22,  1.13it/s]

[Error] 요청 실패 (index: 983, URL: https://stat.molit.go.kr/portal/cate/statView.do?hRsId=57&hFormId=1214&hDivEng=&month_yn=): 500 Server Error: Internal Server Error for url: https://stat.molit.go.kr/portal/cate/statView.do?hRsId=57&hFormId=1214&hDivEng=&month_yn=


100%|██████████| 832/832 [14:59<00:00,  1.08s/it]

데이터 수집 완료.


In [41]:
df_detail_filtered['styleNum'] = style_list
df_detail_filtered['dt_list'] = final_dt_list

In [42]:
df_detail_filtered

,codeCate,rsid,name,url,hFormId,styleNum,dt_list
0,0120000,19,면적별 건축물 현황태극마크,https://stat.molit.go.kr/portal/cate/viewChk.d...,540,94,"[2025, 2024, 2023, 2022, 2021, 2020, 2019, 201..."
1,0120000,19,면적별 건축물 현황(50만_이상_도시)태극마크,https://stat.molit.go.kr/portal/cate/viewChk.d...,5391,1,"[2025, 2024, 2023, 2022, 2021, 2020, 2019, 201..."
2,0120000,19,소유구분별 건축물 현황태극마크,https://stat.molit.go.kr/portal/cate/viewChk.d...,560,95,"[2025, 2024, 2023, 2022, 2021, 2020, 2019, 201..."
3,0120000,19,소유구분별 건축물 현황(50만_이상_도시)태극마크,https://stat.molit.go.kr/portal/cate/viewChk.d...,5392,1,"[2025, 2024, 2023, 2022, 2021, 2020, 2019, 201..."
4,0120000,19,용도별 건축물 현황태극마크,https://stat.molit.go.kr/portal/cate/viewChk.d...,522,96,"[2025, 2024, 2023, 2022, 2021, 2020, 2019, 201..."
...,...,...,...,...,...,...,...
1159,0180000,68,7-3.가출현황_가출인발견(종료)태극마크,https://stat.molit.go.kr/portal/cate/viewChk.d...,894,1,"[2015, 2014, 2013, 2012, 2011, 2010, 2009, 200..."
1160,0180000,67,건널목사고 상세현황태극마크,https://stat.molit.go.kr/portal/cate/viewChk.d...,876,872,"[2024, 2023, 2022, 2021, 2020, 2019, 2018, 201..."
1161,0180000,67,사상사고 상세현황태극마크,https://stat.molit.go.kr/portal/cate/viewChk.d...,878,1,"[2024, 2023, 2022, 2021, 2020, 2019, 2018, 201..."
1162,0180000,67,열차사고 상세현황태극마크,https://stat.molit.go.kr/portal/cate/viewChk.d...,874,1,"[2024, 2023, 2022, 2021, 2020, 2019, 2018, 201..."


In [ ]:
#@title openfiscal


In [59]:
#@title Worldbank
import wbgapi as wb
# 검색어 없이 전체 메타데이터(지표) 조회
results = wb.series.list()

# DataFrame으로 변환
df = pd.DataFrame(results)

In [60]:
df

,id,value
0,AG.CON.FERT.PT.ZS,Fertilizer consumption (% of fertilizer produc...
1,AG.CON.FERT.ZS,Fertilizer consumption (kilograms per hectare ...
2,AG.LND.AGRI.K2,Agricultural land (sq. km)
3,AG.LND.AGRI.ZS,Agricultural land (% of land area)
4,AG.LND.ARBL.HA,Arable land (hectares)
...,...,...
1481,VC.IDP.NWCV,"Internally displaced persons, new displacement..."
1482,VC.IDP.NWDS,"Internally displaced persons, new displacement..."
1483,VC.IHR.PSRC.FE.P5,"Intentional homicides, female (per 100,000 fem..."
1484,VC.IHR.PSRC.MA.P5,"Intentional homicides, male (per 100,000 male)"


In [155]:
importlib.reload(master_table)
from master_table import build_master_table, fetch_one
master = build_master_table(
    kosis_excel_path="주제별통계.xls",
    include_sources=("kosis", "ecos", "imf", "enara", "molit","worldbank","openfiscal"),
    show_progress=True
)

[kosis]:   0%|          | 0/1 [00:00<?, ?task/s]

[ecos]:   0%|          | 0/1 [00:00<?, ?task/s]

[imf]:   0%|          | 0/1 [00:00<?, ?task/s]

[enara]:   0%|          | 0/1 [00:00<?, ?task/s]

[molit]:   0%|          | 0/1 [00:00<?, ?task/s]

c:\Users\apjh2\VS CODE\corruption\apifunction\apifunction\master_table.py:521: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  if node is None:


[worldbank]:   0%|          | 0/1 [00:00<?, ?task/s]

[openfiscal]:   0%|          | 0/1 [00:00<?, ?task/s]

In [156]:
master[master['source'] == 'enara']

,index,source,table_id,table_name,cycle,start_date,end_date,params
61152,enara:2455:245501,enara,245501,10대 수출품목,Y,None,None,"{'stats_code': '245501', 'indicator_code': '24..."
61153,enara:2455:245502,enara,245502,10대 수입품목,Y,None,None,"{'stats_code': '245502', 'indicator_code': '24..."
61154,enara:1609:160901,enara,160901,112접수건수 및 현장평균도착시간,Y,None,None,"{'stats_code': '160901', 'indicator_code': '16..."
61155,enara:1601:160101,enara,160101,1388 청소년전화 접수현황,Y,None,None,"{'stats_code': '160101', 'indicator_code': '16..."
61156,enara:1728:172801,enara,172801,"1심, 2심 무죄현황",Y,None,None,"{'stats_code': '172801', 'indicator_code': '17..."
...,...,...,...,...,...,...,...,...
62221,enara:8050:805002,enara,805002,안전에 대한 전반적 인식,Y,None,None,"{'stats_code': '805002', 'indicator_code': '80..."
62222,enara:8051:805101,enara,805101,야간보행 안전도,Y,None,None,"{'stats_code': '805101', 'indicator_code': '80..."
62223,enara:8098:809802,enara,809802,토양환경 만족도,Y,None,None,"{'stats_code': '809802', 'indicator_code': '80..."
62224,enara:8030:803002,enara,803002,학교교육 효과,Y,None,None,"{'stats_code': '803002', 'indicator_code': '80..."


In [157]:
import apifunction.enara as enara
import master_table
from master_table import build_master_table, fetch_one
importlib.reload(enara)
importlib.reload(master_table)
fetch_one(master, dataset_id="245502")

,주기,항목코드,항목명,통계표코드,통계표명,단위,출처,주석,값
0,1996,T01,품목명,245502,10대 수입품목,백만불,수입통관자료,수입품목 중 상위 10개 품목 통계,NaN
1,1997,T01,품목명,245502,10대 수입품목,백만불,수입통관자료,수입품목 중 상위 10개 품목 통계,NaN
2,1998,T01,품목명,245502,10대 수입품목,백만불,수입통관자료,수입품목 중 상위 10개 품목 통계,NaN
3,1999,T01,품목명,245502,10대 수입품목,백만불,수입통관자료,수입품목 중 상위 10개 품목 통계,NaN
4,2000,T01,품목명,245502,10대 수입품목,백만불,수입통관자료,수입품목 중 상위 10개 품목 통계,NaN
...,...,...,...,...,...,...,...,...,...
691,2020,T02,금액,245502,10대 수입품목,백만불,수입통관자료,수입품목 중 상위 10개 품목 통계,42.8
692,2021,T02,금액,245502,10대 수입품목,백만불,수입통관자료,수입품목 중 상위 10개 품목 통계,45.0
693,2022,T02,금액,245502,10대 수입품목,백만불,수입통관자료,수입품목 중 상위 10개 품목 통계,51.9
694,2023,T02,금액,245502,10대 수입품목,백만불,수입통관자료,수입품목 중 상위 10개 품목 통계,49.1


In [151]:
importlib.reload(master_table)

from master_table import load_table
master = load_table()

In [4]:
import requests

url = "https://www.openfiscaldata.go.kr/op/ko/sd/dtsStats/selectScolSrchList.do"

headers = {
    "AJAX": "true",
    "Accept": "*/*",
    "Content-Type": "application/json",
    "Origin": "https://www.openfiscaldata.go.kr",
    "Referer": "https://www.openfiscaldata.go.kr/op/ko/sd/UOPKOSDA01",
    "User-Agent": "Mozilla/5.0",
}

# cURL의 --data-raw JSON 그대로
payload = {
    "opKoSdDtsStatsDVO": {
        "searchKeyword": "",
        "ofdMngOgNm": "",
        "ognSysNm": "",
        "rlsSvIxNm": "A"
    }
}

r = requests.post(url, headers=headers, json=payload, timeout=30)
r.raise_for_status()

data = r.json()

# 응답 구조가 보통 이렇게 옴
rows = data.get("selectScolSrchList", [])
odt_ids = [row.get("odtId") for row in rows if row.get("odtId")]

In [13]:
import pandas as pd
target_cols = [
    "odtNm",
    "odtId",
    "dtaLoadPrdNm",
    "allDtaClsNm",
    "ofdMngOgCd",
    "ofdMngOgNm",
]
# 필요한 컬럼만 추출해서 DataFrame 생성
df_openfiscal = pd.DataFrame([{col: row.get(col) for col in target_cols} for row in rows])
# (선택) 컬럼 순서 강제
df_openfiscal = df_openfiscal[target_cols]
# (선택) odtId 없는 행 제거
df_openfiscal = df_openfiscal.dropna(subset=["odtId"]).reset_index(drop=True)

base_url = "https://openapi.openfiscaldata.go.kr"
df_openfiscal["href"] = (
    base_url + "/" + df_openfiscal["odtId"].astype(str).str.strip()
)

In [ ]:
import requests

BASE = "https://www.openfiscaldata.go.kr"

def post_json(session, url, payload, headers=None):
    h = {
        "AJAX": "true",
        "Content-Type": "application/json",
        "Referer": f"{BASE}/op/ko/sd/UOPKOSDA01",
        "Origin": BASE,
        "User-Agent": "Mozilla/5.0",
    }
    if headers:
        h.update(headers)
    r = session.post(url, json=payload, headers=h, timeout=30)
    r.raise_for_status()
    return r.json()

def get_openapi_req_vars(session, odt_id: str):
    # 1) odtId 상세 조회 -> srvInfoList 획득
    detail = post_json(
        session,
        f"{BASE}/op/ko/sd/dtsStats/selectSrvDtlInfoList.do",
        {"opKoSdDtsStatsDVO": {"odtId": odt_id}},
    )

    srv_info = detail.get("selectSrvInfoList", [])
    a_srv = next((x for x in srv_info if x.get("rlsSvTyCd") == "A"), None)
    if not a_srv:
        return None, []  # Open API 미제공 odtId

    odt_sv_seq = a_srv.get("odtSvSeq")

    # 2) Open API 메타 조회 -> 요청인자 목록
    payload = {
        "odtId": odt_id,
        "rlsSvTyCd": "A",
        "odtSvSeq": odt_sv_seq,
        # 선택
        # "odtNm": detail.get("selectSrvDtlInfo", {}).get("possDtaNm", ""),
        # "paramVO": {"odtNm": detail.get("selectSrvDtlInfo", {}).get("possDtaNm", "")},
    }

    meta = post_json(
        session,
        f"{BASE}/op/ko/sd/dtsStatsAcol/selectAcolViewList.do",
        payload,
    )

    req_vars = meta.get("selectApiReqVarList", [])
    return odt_sv_seq, req_vars


# 사용 예시
s = requests.Session()
odt_id = "68EHHHFZH403WYI43Q3GM8YQ0"
odt_sv_seq, req_vars = get_openapi_req_vars(s, odt_id)

odtSvSeq: 2
요청인자 수: 1
ACNT_YR String 회계연도


In [ ]:
import pandas as pd
import requests

# 이미 만든 함수 사용:
# get_openapi_req_vars(session, odt_id) -> (odt_sv_seq, req_vars)

def enrich_openfiscal_with_req_vars(df_openfiscal: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    s = requests.Session()

    df = df_openfiscal.copy()
    unique_ids = (
        df["odtId"].dropna().astype(str).str.strip().unique().tolist()
    )

    meta_rows = []   # odtId별 요약(요청인자명 리스트 등)
    req_rows = []    # odtId-요청인자 단위 long table
    err_rows = []    # 실패 로그

    for odt_id in unique_ids:
        try:
            odt_sv_seq, req_vars = get_openapi_req_vars(s, odt_id)

            ds_cols = [v.get("dsColId") for v in req_vars if v.get("dsColId")]
            req_types = [v.get("reqType") for v in req_vars if v.get("reqType")]

            meta_rows.append({
                "odtId": odt_id,
                "odtSvSeq": odt_sv_seq,
                "req_var_count": len(req_vars),
                "req_var_names": ",".join(ds_cols),   # 예: ACNT_YR,OFFC_CD,...
                "req_types": ",".join(sorted(set(req_types))),
                "has_ACNT_YR": "ACNT_YR" in ds_cols,
            })

            for v in req_vars:
                req_rows.append({
                    "odtId": odt_id,
                    "odtSvSeq": odt_sv_seq,
                    "dsColId": v.get("dsColId"),
                    "reqType": v.get("reqType"),
                    "colEpl": v.get("colEpl"),
                })

        except Exception as e:
            err_rows.append({"odtId": odt_id, "error": str(e)})

    df_meta = pd.DataFrame(meta_rows)
    df_reqvars = pd.DataFrame(req_rows)
    df_errors = pd.DataFrame(err_rows)

    # 원본 df_openfiscal에 odtId 기준으로 요청인자 요약 붙이기
    df_enriched = df.merge(df_meta, on="odtId", how="left")

    return df_enriched, df_reqvars, df_errors


# 사용 예시
df_openfiscal_enriched, df_openfiscal_reqvars, df_openfiscal_reqvars_errors = enrich_openfiscal_with_req_vars(df_openfiscal)

           odtNm                      odtId dtaLoadPrdNm  \
0  월별 국가채무(중앙정부)  R83QZS6DVFIK6JV36NP614173           월간   
1      월별 지출운용상황  J4767STHIN5GDO6V457576815           월간   
2       국가채무(D1)  11UN000D6M7UJ2LFRO76OG35Y           연간   
3    공공부문 부채(D3)  415EFE7TWXE4SE40IE8E4JJA7           연간   
4       분야별출자금추이  CR1335G515AW73XVX75034WWS           연간   

                allDtaClsNm ofdMngOgCd   ofdMngOgNm  \
0        국문▶ 중앙정부>월간재정동향>채무    1312338  기획재정부 재정정책국   
1     국문▶ 중앙정부>상세재정통계>집행>지출    1051000        기획재정부   
2      국문▶ 중앙정부>상세재정통계>국가채무    1312338  기획재정부 재정정책국   
3      국문▶ 중앙정부>상세재정통계>국가채무    1312338  기획재정부 재정정책국   
4  국문▶ 중앙정부>주요재정통계>재정지출>유형별    B553658      한국재정정보원   

                                                href odtSvSeq  req_var_count  \
0  https://openapi.openfiscaldata.go.kr/R83QZS6DV...        3              2   
1  https://openapi.openfiscaldata.go.kr/J4767STHI...        2              4   
2  https://openapi.openfiscaldata.go.kr/11UN000D6...        3       

In [ ]:
# 1) reqType에서 "필수"만 남김 (값 형태는 환경에 맞게 추가)
required_tokens = {"필수", "Y", "R", "REQUIRED", "MANDATORY"}
req_required = df_openfiscal_reqvars[
    df_openfiscal_reqvars["reqType"]
    .fillna("")
    .astype(str)
    .str.upper()
    .isin({t.upper() for t in required_tokens})
].copy()
# 2) odtId별 필수 인자 목록만 요약
req_required_summary = (
    req_required
    .groupby("odtId", as_index=False)
    .agg(
        required_var_names=("dsColId", lambda s: ",".join(sorted(set(x for x in s if pd.notna(x))))),
        required_var_count=("dsColId", lambda s: len(set(x for x in s if pd.notna(x))))
    )
)
# 3) 원본 df_openfiscal에 left merge -> 행 수 유지
before_n = len(df_openfiscal)
df_openfiscal = df_openfiscal.merge(req_required_summary, on="odtId", how="left")
df_openfiscal["required_var_names"] = df_openfiscal["required_var_names"].fillna("")
df_openfiscal["required_var_count"] = df_openfiscal["required_var_count"].fillna(0).astype(int)

In [28]:
df_openfiscal

,odtNm,odtId,dtaLoadPrdNm,allDtaClsNm,ofdMngOgCd,ofdMngOgNm,href,required_var_names,required_var_count
0,월별 국가채무(중앙정부),R83QZS6DVFIK6JV36NP614173,월간,국문▶ 중앙정부>월간재정동향>채무,1312338,기획재정부 재정정책국,https://openapi.openfiscaldata.go.kr/R83QZS6DV...,,0
1,월별 지출운용상황,J4767STHIN5GDO6V457576815,월간,국문▶ 중앙정부>상세재정통계>집행>지출,1051000,기획재정부,https://openapi.openfiscaldata.go.kr/J4767STHI...,,0
2,국가채무(D1),11UN000D6M7UJ2LFRO76OG35Y,연간,국문▶ 중앙정부>상세재정통계>국가채무,1312338,기획재정부 재정정책국,https://openapi.openfiscaldata.go.kr/11UN000D6...,,0
3,공공부문 부채(D3),415EFE7TWXE4SE40IE8E4JJA7,연간,국문▶ 중앙정부>상세재정통계>국가채무,1312338,기획재정부 재정정책국,https://openapi.openfiscaldata.go.kr/415EFE7TW...,,0
4,분야별출자금추이,CR1335G515AW73XVX75034WWS,연간,국문▶ 중앙정부>주요재정통계>재정지출>유형별,B553658,한국재정정보원,https://openapi.openfiscaldata.go.kr/CR1335G51...,,0
...,...,...,...,...,...,...,...,...,...
119,분야별이월추이,45D7DP4ZKR53QY6IT0T6S32M4,연간,국문▶ 중앙정부>주요재정통계>재정지출>예산의집행(총지출기준),B553658,한국재정정보원,https://openapi.openfiscaldata.go.kr/45D7DP4ZK...,,0
120,성질별국가채무추이,4QX5DE62F3X01M6Z20P7BUI15,연간,국문▶ 중앙정부>주요재정통계>재정건전성>국가채무(D1),B553658,한국재정정보원,https://openapi.openfiscaldata.go.kr/4QX5DE62F...,,0
121,성질별집행추이,7L8O1IGZ3S7S2873G54C75F3U,연간,국문▶ 중앙정부>주요재정통계>재정지출>예산의집행(총지출기준),B553658,한국재정정보원,https://openapi.openfiscaldata.go.kr/7L8O1IGZ3...,,0
122,중앙관서별총지출추이,WHFDLEJ0WYTNZ1RJ1Z466IH61,연간,국문▶ 중앙정부>주요재정통계>재정지출>소관별,B553658,한국재정정보원,https://openapi.openfiscaldata.go.kr/WHFDLEJ0W...,,0


In [159]:
master.to_csv('master.csv',encoding='utf-8-sig')